# Validation finale — Analyse comparative CPU TSA LUT vs FPGA THESE vfin

Ce notebook compare uniquement les deux campagnes CPU TSA LUT et FPGA THESE vfin,
reconstruit une table winner complete, et produit les metriques et graphiques
necessaires a la validation comparative finale.

In [ ]:
# ==========================================================
# CELLULE 1 — CHARGEMENT, FUSION
# ==========================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    Markdown = lambda text: text

# ---------- Configuration fixe : comparaison CPU TSA LUT vs FPGA ----------
CPU_LUT_DIR = Path("Resultats validation TSA") / "campagne_cpu_tsa_lut"
FPGA_DIR    = Path("Resultats validation TSA") / "campagne_fpga_these_vfin"
OUT_DIR     = Path("Resultats validation TSA") / "analyse_cpu_tsa_lut_vs_fpga_these_vfin"

PREFIX_CPU_LUT = "validation_cpu_tsa"
PREFIX_FPGA    = "validation_fpga_these_vfin"
PFX = "cpu_tsa_lut_vs_fpga_these_vfin"
EPS = 1e-30

ML = {
    "cpu_tsa_lut": "CPU TSA LUT",
    "fpga": "FPGA THESE vfin",
}


def _resolve_input_file(base_dir: Path, prefix: str, kind: str, fallback_glob="*raw_results.csv"):
    """Resout le fichier input dans base_dir avec prefix donne."""
    base_dir = Path(base_dir)  # Assure que c'est un Path
    f = base_dir / f"{prefix}_{kind}.csv"
    if f.exists():
        return f
    # Fallback glob uniquement si le fichier n'existe pas
    if kind in ["raw_results", "winner_results"]:
        pat = f"*{kind}.csv" if kind == "winner_results" else fallback_glob
        cands = sorted(base_dir.glob(pat))
        if cands:
            used = cands[0]
            print(f"[WARN] Fichier {prefix}_{kind}.csv absent. Utilisation de: {used.name}")
            return used
    raise FileNotFoundError(f"Fichier introuvable: {f}")


def _normalize_raw_columns(df: pd.DataFrame, method_fallback: str) -> pd.DataFrame:
    df = df.copy()
    if "method" not in df.columns:
        df["method"] = method_fallback
    if "prn_tested" not in df.columns and "prn_winner" in df.columns:
        df["prn_tested"] = df["prn_winner"]
    if "prn_winner" not in df.columns and "prn_tested" in df.columns:
        df["prn_winner"] = df["prn_tested"]
    if "status" not in df.columns:
        df["status"] = "ok"
    if "error_message" not in df.columns:
        df["error_message"] = ""
    if "time_s" not in df.columns:
        for c in ["time_kernel_s", "time_end_to_end_s"]:
            if c in df.columns:
                df["time_s"] = df[c]
                break
        if "time_s" not in df.columns:
            df["time_s"] = np.nan
    if "peak_ratio" not in df.columns:
        if "peak" in df.columns and "mean_power" in df.columns:
            den = np.maximum(df["mean_power"].astype(float).to_numpy(), EPS)
            df["peak_ratio"] = df["peak"].astype(float).to_numpy() / den
        else:
            df["peak_ratio"] = np.nan
    required = [
        "file", "method", "status", "error_message", "snr_db", "prn_injected", "prn_tested",
        "doppler_true_hz", "phase_true_chip", "doppler_meas_hz", "phase_meas_chip",
        "peak", "peak_ratio", "time_s",
    ]
    miss = [c for c in required if c not in df.columns]
    if miss:
        raise ValueError(f"Colonnes manquantes apres harmonisation: {miss}")
    return df


def _compute_winner_margin(raw_df: pd.DataFrame) -> pd.Series:
    """Calcule winner_margin_db (1er pic / 2e pic) par fichier+methode depuis le raw."""
    margins = {}
    for (fn, meth), grp in raw_df[raw_df["status"] == "ok"].groupby(["file", "method"]):
        ranked = grp.sort_values("peak_ratio", ascending=False).reset_index(drop=True)
        best = float(ranked.iloc[0]["peak_ratio"]) if len(ranked) >= 1 else EPS
        second = float(ranked.iloc[1]["peak_ratio"]) if len(ranked) >= 2 else EPS
        ratio = best / max(second, EPS)
        margins[(fn, meth)] = 10.0 * np.log10(max(ratio, EPS))
    return margins


def _normalize_winner_columns(df: pd.DataFrame, method_tag: str, margin_map: dict) -> pd.DataFrame:
    """Harmonise le winner CSV issu de la campagne (décision de l'algorithme)."""
    df = df.copy()
    if "method" not in df.columns:
        df["method"] = method_tag
    if "status" not in df.columns:
        df["status"] = "ok"
    if "error_message" not in df.columns:
        df["error_message"] = ""
    if "prn_winner" not in df.columns and "prn_tested" in df.columns:
        df["prn_winner"] = df["prn_tested"]
    if "prn_tested" not in df.columns and "prn_winner" in df.columns:
        df["prn_tested"] = df["prn_winner"]
    if "doppler_err_hz" not in df.columns:
        df["doppler_err_hz"] = df["doppler_meas_hz"] - df["doppler_true_hz"]
    if "phase_err_chip" not in df.columns:
        nb = 1023
        diff = (df["phase_meas_chip"].astype(int) - df["phase_true_chip"].astype(int)) % nb
        df["phase_err_chip"] = diff.clip(upper=nb // 2)
    if "winner_margin_db" not in df.columns:
        df["winner_margin_db"] = df.apply(
            lambda r: margin_map.get((r["file"], r["method"]), np.nan), axis=1
        )
    # Flags basés uniquement sur la décision de l'algorithme
    prn_ok = df["prn_winner"].astype(int) == df["prn_injected"].astype(int)
    if "strict_success" not in df.columns:
        df["strict_success"] = prn_ok.astype(int)
    if "pd_flag" not in df.columns:
        df["pd_flag"] = df["strict_success"]
    if "pfa_flag" not in df.columns:
        df["pfa_flag"] = (~prn_ok).astype(int)
    if "pm_flag" not in df.columns:
        df["pm_flag"] = (prn_ok & (df["strict_success"] == 0)).astype(int)
    return df


OUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCES = {
    "cpu_tsa_lut": (CPU_LUT_DIR, PREFIX_CPU_LUT, "cpu_tsa_lut"),
    "fpga":        (FPGA_DIR,    PREFIX_FPGA,    "fpga"),
}
selected_methods = ["cpu_tsa_lut", "fpga"]

COMPARISON_LABELS = [ML.get(m, m) for m in selected_methods]
COMPARISON_TITLE  = " vs ".join(COMPARISON_LABELS)

# --- Chargement raw ---
raw_parts = []
loaded_files = {}
for meth in selected_methods:
    base_dir, prefix, fallback_method = SOURCES[meth]
    f = _resolve_input_file(base_dir, prefix, "raw_results")
    loaded_files[meth] = str(f)
    df = _normalize_raw_columns(pd.read_csv(f), fallback_method)
    df["method"] = meth
    raw_parts.append(df)

raw = pd.concat(raw_parts, ignore_index=True)

# --- Calcul winner_margin_db depuis le raw (utilisé si absent du winner CSV) ---
margin_map = _compute_winner_margin(raw)

# --- Chargement winner (décision native de l'algorithme) ---
winner_parts = []
for meth in selected_methods:
    base_dir, prefix, _ = SOURCES[meth]
    f = _resolve_input_file(base_dir, prefix, "winner_results")
    df = _normalize_winner_columns(pd.read_csv(f), meth, margin_map)
    df["method"] = meth
    winner_parts.append(df)

W = pd.concat(winner_parts, ignore_index=True)
W.to_csv(OUT_DIR / f"{PFX}_winner_results.csv", index=False)
raw.to_csv(OUT_DIR / f"{PFX}_raw_results.csv", index=False)

ok = W[W["status"] == "ok"].copy()
rok = raw[raw["status"] == "ok"].copy()
rok["is_decoy"] = (rok["prn_tested"].astype(int) != rok["prn_injected"].astype(int)).astype(int)
rok_dec = rok[rok["is_decoy"] == 1].copy()
methods = selected_methods.copy()

print("=== SOURCES CHARGEES ===")
for meth in selected_methods:
    print(f"- {ML.get(meth, meth)}: {loaded_files[meth]}")
print(f"- Export analyse: {OUT_DIR}")
print(f"- Comparaison: {COMPARISON_TITLE}")
print(f"\nWinner charges: {len(W)}")
print(f"Lignes raw chargees: {len(raw)}")
display(Markdown(f"**Comparaison active : {COMPARISON_TITLE}**"))


## Hypotheses

- Les exports CPU TSA LUT et FPGA suivent le schema `*_raw_results.csv`.
- Si le prefixe configure n'existe pas, la cellule 1 prend automatiquement le premier `*raw_results.csv` disponible dans le dossier cible.
- Si `method` est absent, il est force a `cpu_tsa_lut` ou `fpga`.
- Si `prn_tested` est absent mais `prn_winner` existe, il est reconstruit automatiquement.
- Si certaines colonnes temps sont absentes, des fallbacks (`time_s`, `time_kernel_s`, `time_end_to_end_s`) sont appliques.

In [ ]:
# ==========================================================
# CELLULE 2 — PARAMETRES DE CAMPAGNE
# ==========================================================
campaign_params = pd.DataFrame({
    "Parametre": [
        "Frequence echantillonnage (fs)",
        "Frequence intermediaire (IF)",
        "Nombre echantillons (N)",
        "Nombre phases code (nb_phases)",
        "Plage Doppler",
        "Pas Doppler (fd_step)",
        "Nb frequences Doppler",
        "Seuil detection",
        "PRN injecte",
        "PRN leurres par signal",
        "Nombre de fichiers signal",
        "Plage SNR",
        "Methodes comparees",
    ],
    "Valeur": [
        "11.999 MHz",
        "3.563 MHz",
        str(int(rok.iloc[0].get("n_samples", 11999))) if "n_samples" in rok.columns else "11999",
        "1023",
        f"[{int(rok['doppler_true_hz'].min())}, {int(rok['doppler_true_hz'].max())}] Hz",
        f"{int(rok['doppler_true_hz'].astype(float).diff().abs().dropna().mode().iloc[0])} Hz" if rok['doppler_true_hz'].nunique() > 1 else "N/A",
        str(rok['doppler_true_hz'].astype(int).nunique()),
        "Defini par l'algorithme",
        str(sorted(rok['prn_injected'].astype(int).unique())),
        str(rok.groupby(['file', 'method']).size().iloc[0] - 1),
        str(ok['file'].nunique()),
        f"{sorted(ok['snr_db'].unique())}",
        COMPARISON_TITLE if 'COMPARISON_TITLE' in globals() else 'CPU TSA LUT vs FPGA STSA vfin',
    ]
})
campaign_params.to_csv(OUT_DIR / f"{PFX}_campaign_params.csv", index=False)
print("=== PARAMETRES DE CAMPAGNE ===")
display(campaign_params)


In [ ]:
# ==========================================================
# CELLULE 3 — TABLEAU GLOBAL PAR METHODE
# ==========================================================
tg = (
    ok.groupby("method", as_index=False)
    .agg(
        n_signaux=("file", "nunique"),
        n_acquisitions=("file", "count"),
        succes_strict_pct=("strict_success", lambda s: 100.0*s.mean()),
        Pd_pct=("pd_flag", lambda s: 100.0*s.mean()),
        Pfa_pct=("pfa_flag", lambda s: 100.0*s.mean()),
        Pm_pct=("pm_flag", lambda s: 100.0*s.mean()),
        doppler_mae_hz=("doppler_err_hz", lambda s: np.mean(np.abs(s))),
        doppler_max_hz=("doppler_err_hz", "max"),
        phase_mae_chip=("phase_err_chip", lambda s: np.mean(np.abs(s))),
        phase_max_chip=("phase_err_chip", "max"),
        peak_ratio_mean=("peak_ratio", "mean"),
        peak_ratio_std=("peak_ratio", "std"),
        winner_margin_mean_db=("winner_margin_db", "mean"),
        winner_margin_min_db=("winner_margin_db", "min"),
        time_mean_ms=("time_s", lambda s: 1e3*s.mean()),
        time_median_ms=("time_s", lambda s: 1e3*s.median()),
    ).sort_values("method").reset_index(drop=True)
)
tg["label"] = tg["method"].map(ML)
tg.to_csv(OUT_DIR / f"{PFX}_table_global.csv", index=False)
print("=== TABLEAU GLOBAL PAR METHODE ===")
display(tg.round(4))


In [ ]:
# ==========================================================
# CELLULE 4 — TABLEAU PAR SNR
# ==========================================================
ts = (
    ok.groupby(["method", "snr_db"], as_index=False)
    .agg(
        n=("file", "count"),
        succes_strict_pct=("strict_success", lambda s: 100.0*s.mean()),
        Pd_pct=("pd_flag", lambda s: 100.0*s.mean()),
        Pfa_pct=("pfa_flag", lambda s: 100.0*s.mean()),
        Pm_pct=("pm_flag", lambda s: 100.0*s.mean()),
        doppler_mae_hz=("doppler_err_hz", lambda s: np.mean(np.abs(s))),
        phase_mae_chip=("phase_err_chip", lambda s: np.mean(np.abs(s))),
        peak_ratio_mean=("peak_ratio", "mean"),
        winner_margin_mean_db=("winner_margin_db", "mean"),
        time_mean_ms=("time_s", lambda s: 1e3*s.mean()),
    ).sort_values(["snr_db", "method"]).reset_index(drop=True)
)
ts["label"] = ts["method"].map(ML)
ts.to_csv(OUT_DIR / f"{PFX}_table_by_snr.csv", index=False)
print("=== TABLEAU PAR SNR ===")
display(ts.round(4))


In [ ]:
# ==========================================================
# CELLULE 5 — ANALYSE PFA SUR PRN LEURRES
# ==========================================================
rok_inj  = rok[rok["is_decoy"] == 0].copy()
rok_dec  = rok[rok["is_decoy"] == 1].copy()

ratio_min = CFG["detection_ratio_min"]
rok_dec["false_detection"] = (rok_dec["peak_ratio"].astype(float) >= ratio_min).astype(int)

# Volume
vol = rok.groupby(["method", "is_decoy"], as_index=False).size().rename(columns={"size": "n"})
vol["type"] = vol["is_decoy"].map({0: "PRN injecte", 1: "PRN leurre"})
print("=== VOLUME INJECTE vs LEURRES ===")
display(vol[["method", "type", "n"]])

# PFA globale sur leurres
pfa_global = (
    rok_dec.groupby("method", as_index=False)
    .agg(
        n_leurres=("file", "count"),
        n_fausses_detections=("false_detection", "sum"),
        Pfa_leurres_pct=("false_detection", lambda s: 100.0*s.mean()),
        peak_ratio_leurre_mean=("peak_ratio", "mean"),
        peak_ratio_leurre_max=("peak_ratio", "max"),
    ).sort_values("method").reset_index(drop=True)
)
pfa_global["label"] = pfa_global["method"].map(ML)
pfa_global.to_csv(OUT_DIR / f"{PFX}_pfa_leurres_global.csv", index=False)
print("\n=== PFA GLOBALE SUR LEURRES ===")
display(pfa_global.round(4))

# PFA par SNR
rok_dec["snr_db"] = rok_dec["snr_db"].astype(float)
pfa_snr = (
    rok_dec.groupby(["method", "snr_db"], as_index=False)
    .agg(
        n_leurres=("file", "count"),
        n_fausses_det=("false_detection", "sum"),
        Pfa_pct=("false_detection", lambda s: 100.0*s.mean()),
        peak_ratio_max=("peak_ratio", "max"),
    ).sort_values(["snr_db", "method"]).reset_index(drop=True)
)
pfa_snr["label"] = pfa_snr["method"].map(ML)
pfa_snr.to_csv(OUT_DIR / f"{PFX}_pfa_by_snr.csv", index=False)
print("\n=== PFA PAR SNR (sur leurres) ===")
display(pfa_snr.round(4))

# PFA par PRN leurre
pfa_prn = (
    rok_dec.groupby(["method", "prn_tested"], as_index=False)
    .agg(
        n=("file", "count"),
        n_false=("false_detection", "sum"),
        Pfa_pct=("false_detection", lambda s: 100.0*s.mean()),
        peak_ratio_mean=("peak_ratio", "mean"),
    ).sort_values(["method", "Pfa_pct"], ascending=[True, False]).reset_index(drop=True)
)
pfa_prn.to_csv(OUT_DIR / f"{PFX}_pfa_by_prn.csv", index=False)
print("\n=== PFA PAR PRN LEURRE ===")
display(pfa_prn.round(4))

In [ ]:
# ==========================================================
# CELLULE 6 — CONCORDANCE CPU vs FPGA
# ==========================================================
methods = sorted(ok["method"].unique())
pcols = ["strict_success", "prn_winner", "doppler_meas_hz", "phase_meas_chip",
         "peak_ratio", "detected", "doppler_err_hz", "phase_err_chip",
         "winner_margin_db"]
pcols_avail = [c for c in pcols if c in ok.columns]
wide = ok.pivot(index="file", columns="method", values=pcols_avail)

agr = []
for i, ml in enumerate(methods):
    for mr in methods[i+1:]:
        cols = []
        for c in pcols_avail:
            cols.extend([(c, ml), (c, mr)])
        cm = wide[[c for c in cols if c in wide.columns]].dropna()
        if cm.empty:
            continue
        row = {
            "gauche": ML.get(ml, ml), "droite": ML.get(mr, mr),
            "n_commun": len(cm),
            "accord_decision_pct": 100.0*np.mean(cm[("strict_success",ml)] == cm[("strict_success",mr)]),
            "accord_prn_pct": 100.0*np.mean(cm[("prn_winner",ml)] == cm[("prn_winner",mr)]),
            "accord_doppler_exact_pct": 100.0*np.mean(cm[("doppler_meas_hz",ml)] == cm[("doppler_meas_hz",mr)]),
            "accord_phase_exact_pct": 100.0*np.mean(cm[("phase_meas_chip",ml)] == cm[("phase_meas_chip",mr)]),
            "delta_doppler_moy_hz": float(np.mean(np.abs(cm[("doppler_meas_hz",ml)] - cm[("doppler_meas_hz",mr)]))),
            "delta_phase_moy_chip": float(np.mean(np.abs(cm[("phase_meas_chip",ml)] - cm[("phase_meas_chip",mr)]))),
            "delta_peak_ratio_moy": float(np.mean(np.abs(cm[("peak_ratio",ml)] - cm[("peak_ratio",mr)]))),
            "delta_peak_ratio_max": float(np.max(np.abs(cm[("peak_ratio",ml)] - cm[("peak_ratio",mr)]))),
            "delta_margin_moy_db": float(np.mean(np.abs(cm[("winner_margin_db",ml)] - cm[("winner_margin_db",mr)]))) if ("winner_margin_db",ml) in cm.columns else np.nan,
            "correlation_peak_ratio": float(cm[("peak_ratio",ml)].corr(cm[("peak_ratio",mr)])),
        }
        agr.append(row)
agr_df = pd.DataFrame(agr)
agr_df.to_csv(OUT_DIR / f"{PFX}_concordance.csv", index=False)

print("=== CONCORDANCE CPU TSA vs FPGA IP ===")
display(agr_df.round(4))

# Concordance par SNR (pairwise, compatible 2 ou 3 methodes)
agr_snr_rows = []
if len(methods) >= 2:
    for i in range(len(methods)):
        for j in range(i + 1, len(methods)):
            ml, mr = methods[i], methods[j]
            for snr in sorted(ok["snr_db"].unique()):
                oksnr = ok[ok["snr_db"] == snr]
                ws = oksnr.pivot(index="file", columns="method", values=["strict_success", "doppler_meas_hz", "phase_meas_chip", "peak_ratio"]).dropna()
                if ws.empty or ("strict_success", ml) not in ws.columns or ("strict_success", mr) not in ws.columns:
                    continue
                agr_snr_rows.append({
                    "gauche": ML.get(ml, ml),
                    "droite": ML.get(mr, mr),
                    "snr_db": snr,
                    "n": len(ws),
                    "accord_decision_pct": 100.0 * np.mean(ws[("strict_success", ml)] == ws[("strict_success", mr)]),
                    "accord_doppler_pct": 100.0 * np.mean(ws[("doppler_meas_hz", ml)] == ws[("doppler_meas_hz", mr)]),
                    "accord_phase_pct": 100.0 * np.mean(ws[("phase_meas_chip", ml)] == ws[("phase_meas_chip", mr)]),
                    "delta_peak_ratio_moy": float(np.mean(np.abs(ws[("peak_ratio", ml)] - ws[("peak_ratio", mr)]))),
                })

agr_snr_df = pd.DataFrame(agr_snr_rows)
agr_snr_df.to_csv(OUT_DIR / f"{PFX}_concordance_by_snr.csv", index=False)
print("\n=== CONCORDANCE PAR SNR ===")
display(agr_snr_df.round(4))

In [ ]:
# ==========================================================
# CELLULE 7 — CAS DIFFICILES (top 20)
# ==========================================================
hc = ok.sort_values(
    ["strict_success", "winner_margin_db", "peak_ratio", "doppler_err_hz", "phase_err_chip"],
    ascending=[True, True, True, False, False],
).head(20)
hc_cols = [
    "file", "method", "snr_db", "prn_injected", "prn_winner",
    "pd_flag", "pfa_flag", "pm_flag",
    "doppler_true_hz", "doppler_meas_hz", "doppler_err_hz",
    "phase_true_chip", "phase_meas_chip", "phase_err_chip",
    "peak_ratio", "second_peak_ratio", "winner_margin_db",
    "strict_success",
]
hc = hc[[c for c in hc_cols if c in hc.columns]]
hc.to_csv(OUT_DIR / f"{PFX}_hard_cases.csv", index=False)
print("=== CAS DIFFICILES (top 20) ===")
display(hc.round(4))

In [ ]:
# ==========================================================
# CELLULE 8 — COURBES Pd / Pfa / Pm vs SNR
# ==========================================================
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

for meth, g in ts.groupby("method"):
    lb = ML.get(meth, meth)
    axes[0].plot(g["snr_db"], g["Pd_pct"], marker="o", lw=2, label=f"{lb} Pd")
    axes[0].plot(g["snr_db"], g["Pfa_pct"], marker="x", ls="--", lw=1.5, label=f"{lb} Pfa")
    if g["Pm_pct"].sum() > 0:
        axes[0].plot(g["snr_db"], g["Pm_pct"], marker="s", ls=":", lw=1.5, label=f"{lb} Pm")
    axes[1].plot(g["snr_db"], g["doppler_mae_hz"], marker="o", lw=2, label=lb)
    axes[2].plot(g["snr_db"], g["phase_mae_chip"], marker="o", lw=2, label=lb)

axes[0].set_title("Detection (Pd), Fausse alarme (Pfa), Miss (Pm) vs SNR")
axes[0].set_xlabel("SNR (dB)"); axes[0].set_ylabel("%")
axes[0].set_ylim(-2, 102); axes[0].grid(alpha=0.3); axes[0].legend(fontsize=7)

axes[1].axhline(CFG["doppler_tol_hz"], color="red", ls="--", alpha=0.7, label="Tolerance")
axes[1].set_title("Erreur Doppler MAE vs SNR")
axes[1].set_xlabel("SNR (dB)"); axes[1].set_ylabel("Hz")
axes[1].grid(alpha=0.3); axes[1].legend(fontsize=8)

axes[2].axhline(CFG["phase_tol_chip"], color="red", ls="--", alpha=0.7, label="Tolerance")
axes[2].set_title("Erreur Phase MAE vs SNR")
axes[2].set_xlabel("SNR (dB)"); axes[2].set_ylabel("chips")
axes[2].grid(alpha=0.3); axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / f"{PFX}_pd_pfa_pm_vs_snr.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ==========================================================
# CELLULE 9 — GRAPHIQUES SEPARES PAR METHODE (4 panneaux)
# ==========================================================
rt_snr = ok.groupby(["method","snr_db"], as_index=False)["time_s"].mean()
rt_snr["time_ms"] = 1e3 * rt_snr["time_s"]

for meth in sorted(ok["method"].unique()):
    lb = ML.get(meth, meth)
    g = ts[ts["method"]==meth].sort_values("snr_db")
    gt = rt_snr[rt_snr["method"]==meth].sort_values("snr_db")
    if g.empty: continue

    fig, ax = plt.subplots(1, 4, figsize=(20, 4.4))
    ax[0].plot(g["snr_db"], g["Pd_pct"], "o-", lw=2, label="Pd")
    ax[0].plot(g["snr_db"], g["Pfa_pct"], "x--", lw=1.5, label="Pfa")
    ax[0].set_title(f"{lb} — Detection"); ax[0].set_ylim(-2,102)
    ax[0].set_xlabel("SNR (dB)"); ax[0].set_ylabel("%")
    ax[0].grid(alpha=.3); ax[0].legend(fontsize=8)

    ax[1].plot(g["snr_db"], g["doppler_mae_hz"], "o-", lw=2, c="#2ca02c")
    ax[1].axhline(CFG["doppler_tol_hz"], c="red", ls="--", alpha=.7, label="Tol")
    ax[1].set_title(f"{lb} — Doppler MAE")
    ax[1].set_xlabel("SNR (dB)"); ax[1].set_ylabel("Hz")
    ax[1].grid(alpha=.3); ax[1].legend(fontsize=8)

    ax[2].plot(g["snr_db"], g["phase_mae_chip"], "o-", lw=2, c="#9467bd")
    ax[2].axhline(CFG["phase_tol_chip"], c="red", ls="--", alpha=.7, label="Tol")
    ax[2].set_title(f"{lb} — Phase MAE")
    ax[2].set_xlabel("SNR (dB)"); ax[2].set_ylabel("chips")
    ax[2].grid(alpha=.3); ax[2].legend(fontsize=8)

    if not gt.empty:
        ax[3].plot(gt["snr_db"], gt["time_ms"], "o-", lw=2, c="#ff7f0e")
    ax[3].set_title(f"{lb} — Temps moyen")
    ax[3].set_xlabel("SNR (dB)"); ax[3].set_ylabel("ms")
    ax[3].grid(alpha=.3)

    fig.suptitle(f"Validation GNSS — {lb}", fontsize=13)
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{PFX}_method_{meth}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ==========================================================
# CELLULE 10 — PFA LEURRES vs SNR (graphiques)
# ==========================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for meth, g in pfa_snr.groupby("method"):
    lb = ML.get(meth, meth)
    axes[0].plot(g["snr_db"], g["Pfa_pct"], marker="o", lw=2, label=lb)
    axes[1].plot(g["snr_db"], g["peak_ratio_max"], marker="s", lw=2, label=lb)

axes[0].set_title("Pfa leurres vs SNR")
axes[0].set_xlabel("SNR (dB)"); axes[0].set_ylabel("Pfa (%)")
axes[0].set_ylim(-2, max(10, pfa_snr["Pfa_pct"].max()*1.2+1))
axes[0].grid(alpha=.3); axes[0].legend()

axes[1].axhline(CFG["detection_ratio_min"], c="red", ls="--", alpha=.7, label=f"Seuil={CFG['detection_ratio_min']}")
axes[1].set_title("Peak ratio max leurre vs SNR")
axes[1].set_xlabel("SNR (dB)"); axes[1].set_ylabel("Peak ratio")
axes[1].grid(alpha=.3); axes[1].legend()

plt.tight_layout()
plt.savefig(OUT_DIR / f"{PFX}_pfa_leurres_vs_snr.png", dpi=150, bbox_inches="tight")
plt.show()

# Barplot PRN injecte vs leurres
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(vol["method"].unique()))
w = 0.35
for i, meth in enumerate(sorted(vol["method"].unique())):
    m = vol[vol["method"]==meth]
    ni = int(m[m["is_decoy"]==0]["n"].values[0]) if len(m[m["is_decoy"]==0])>0 else 0
    nd = int(m[m["is_decoy"]==1]["n"].values[0]) if len(m[m["is_decoy"]==1])>0 else 0
    ax.bar(i - w/2, ni, w, label="Injecte" if i==0 else "", color="#4C78A8")
    ax.bar(i + w/2, nd, w, label="Leurres" if i==0 else "", color="#F58518")
ax.set_xticks(range(len(sorted(vol["method"].unique()))))
ax.set_xticklabels([ML.get(m,m) for m in sorted(vol["method"].unique())])
ax.set_ylabel("Nombre de tests raw")
ax.set_title("Volume PRN injecte vs PRN leurres")
ax.legend(); ax.grid(axis="y", alpha=.3)
plt.tight_layout()
plt.savefig(OUT_DIR / f"{PFX}_volume_inj_vs_leurres.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ==========================================================
# CELLULE 11 — HEATMAPS SNR x DOPPLER (succes strict)
# ==========================================================
for meth in sorted(ok["method"].unique()):
    lb = ML.get(meth, meth)
    m = ok[ok["method"]==meth]
    pv = m.pivot_table(index="snr_db", columns="doppler_true_hz",
                       values="strict_success", aggfunc="mean") * 100.0
    if pv.shape[0]<2 or pv.shape[1]<2:
        print(f"{lb}: pas assez de points pour heatmap"); continue

    fig, ax = plt.subplots(figsize=(max(10, pv.shape[1]*0.55), max(3.5, pv.shape[0]*0.7)))
    im = ax.imshow(pv.values, aspect="auto", cmap="RdYlGn", vmin=0, vmax=100, origin="lower")
    ax.set_xticks(range(pv.shape[1]))
    ax.set_xticklabels([f"{int(d)/1000:.1f}k" if abs(d)>=1000 else str(int(d)) for d in pv.columns],
                       rotation=45, ha="right", fontsize=7)
    ax.set_yticks(range(pv.shape[0]))
    ax.set_yticklabels([f"{int(s)} dB" for s in pv.index], fontsize=8)
    ax.set_xlabel("Doppler injecte (Hz)")
    ax.set_ylabel("SNR (dB)")
    ax.set_title(f"{lb} — Succes strict (%) par SNR x Doppler")
    for r in range(pv.shape[0]):
        for c in range(pv.shape[1]):
            v = pv.values[r, c]
            color = "white" if v < 50 else "black"
            ax.text(c, r, f"{v:.0f}", ha="center", va="center", fontsize=6, color=color)
    plt.colorbar(im, ax=ax, label="Succes (%)")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{PFX}_heatmap_{meth}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ==========================================================
# CELLULE 12 — DISTRIBUTIONS ERREURS (Doppler, Phase, Peak ratio)
# ==========================================================
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

for meth in sorted(ok["method"].unique()):
    lb = ML.get(meth, meth)
    m = ok[ok["method"]==meth]
    axes[0].hist(m["doppler_err_hz"], bins=30, alpha=0.6, label=lb)
    axes[1].hist(m["phase_err_chip"], bins=20, alpha=0.6, label=lb)
    axes[2].hist(m["peak_ratio"], bins=30, alpha=0.6, label=lb)

axes[0].axvline(CFG["doppler_tol_hz"], c="red", ls="--", alpha=.7, label="Tol")
axes[0].set_title("Distribution erreur Doppler")
axes[0].set_xlabel("Erreur (Hz)"); axes[0].legend(fontsize=8); axes[0].grid(alpha=.3)

axes[1].axvline(CFG["phase_tol_chip"], c="red", ls="--", alpha=.7, label="Tol")
axes[1].set_title("Distribution erreur Phase")
axes[1].set_xlabel("Erreur (chips)"); axes[1].legend(fontsize=8); axes[1].grid(alpha=.3)

axes[2].axvline(CFG["detection_ratio_min"], c="red", ls="--", alpha=.7, label="Seuil")
axes[2].set_title("Distribution Peak Ratio (winner)")
axes[2].set_xlabel("Peak Ratio"); axes[2].legend(fontsize=8); axes[2].grid(alpha=.3)

plt.tight_layout()
plt.savefig(OUT_DIR / f"{PFX}_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ==========================================================
# CELLULE 13bis — CONCORDANCE CPU TSA LUT vs FPGA
# ==========================================================

from itertools import combinations

pairs = list(combinations(sorted(ok["method"].unique()), 2))
if not pairs:
    print("Aucune paire a comparer (une seule methode).")
else:
    # --- Concordance pairwise globale ---
    rows_pair = []
    for ml, mr in pairs:
        cols = ["strict_success", "prn_winner", "doppler_meas_hz", "phase_meas_chip", "peak_ratio", "winner_margin_db"]
        cm = ok[ok["method"].isin([ml, mr])].pivot(index="file", columns="method", values=cols).dropna()
        if cm.empty:
            continue
        rows_pair.append({
            "gauche": ML.get(ml, ml),
            "droite": ML.get(mr, mr),
            "n_commun": len(cm),
            "accord_decision_pct": 100.0 * np.mean(cm[("strict_success", ml)] == cm[("strict_success", mr)]),
            "accord_prn_pct": 100.0 * np.mean(cm[("prn_winner", ml)] == cm[("prn_winner", mr)]),
            "accord_doppler_exact_pct": 100.0 * np.mean(cm[("doppler_meas_hz", ml)] == cm[("doppler_meas_hz", mr)]),
            "accord_phase_exact_pct": 100.0 * np.mean(cm[("phase_meas_chip", ml)] == cm[("phase_meas_chip", mr)]),
            "delta_doppler_moy_hz": float(np.mean(np.abs(cm[("doppler_meas_hz", ml)] - cm[("doppler_meas_hz", mr)]))),
            "delta_phase_moy_chip": float(np.mean(np.abs(cm[("phase_meas_chip", ml)] - cm[("phase_meas_chip", mr)]))),
            "delta_peak_ratio_moy": float(np.mean(np.abs(cm[("peak_ratio", ml)] - cm[("peak_ratio", mr)]))),
            "delta_margin_moy_db": float(np.mean(np.abs(cm[("winner_margin_db", ml)] - cm[("winner_margin_db", mr)]))),
        })

    pairwise_df = pd.DataFrame(rows_pair)
    pairwise_df.to_csv(OUT_DIR / f"{PFX}_concordance_pairwise.csv", index=False)
    print("=== CONCORDANCE PAIRWISE (GLOBAL) ===")
    display(pairwise_df.round(4))

    # --- Scatter pairwise ---
    for ml, mr in pairs:
        lbl, lbr = ML.get(ml, ml), ML.get(mr, mr)
        fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
        for k, (col, unit) in enumerate([
            ("peak_ratio", ""),
            ("doppler_meas_hz", " (Hz)"),
            ("phase_meas_chip", " (chips)"),
        ]):
            pv = ok[ok["method"].isin([ml, mr])].pivot(index="file", columns="method", values=col).dropna()
            if ml in pv.columns and mr in pv.columns and len(pv) > 0:
                axes[k].scatter(pv[ml], pv[mr], alpha=0.6, s=25, edgecolors="k", linewidths=0.3)
                vmin = min(pv[ml].min(), pv[mr].min())
                vmax = max(pv[ml].max(), pv[mr].max())
                axes[k].plot([vmin, vmax], [vmin, vmax], "r--", lw=1, alpha=0.7, label="y=x")
                axes[k].set_xlabel(f"{lbl}{unit}")
                axes[k].set_ylabel(f"{lbr}{unit}")
                axes[k].set_title(col.replace("_", " ").title())
                axes[k].grid(alpha=0.3)
                axes[k].legend(fontsize=8)

        fig.suptitle(f"Scatter {lbl} vs {lbr}", y=1.02)
        plt.tight_layout()
        plt.savefig(OUT_DIR / f"{PFX}_scatter_{ml}_vs_{mr}.png", dpi=150, bbox_inches="tight")
        plt.show()

In [ ]:
# ==========================================================
# CELLULE 13 — SCATTER CPU TSA LUT vs FPGA
# ==========================================================
if len(methods) >= 2:
    ml, mr = methods[0], methods[1]
    lbl, lbr = ML.get(ml, ml), ML.get(mr, mr)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

    for i, (col, unit) in enumerate([
        ("peak_ratio", ""), ("doppler_meas_hz", " (Hz)"), ("phase_meas_chip", " (chips)")
    ]):
        pv = ok.pivot(index="file", columns="method", values=col).dropna()
        if ml in pv.columns and mr in pv.columns:
            axes[i].scatter(pv[ml], pv[mr], alpha=0.6, s=25, edgecolors="k", linewidths=0.3)
            vmin = min(pv[ml].min(), pv[mr].min())
            vmax = max(pv[ml].max(), pv[mr].max())
            axes[i].plot([vmin, vmax], [vmin, vmax], "r--", lw=1, alpha=0.7, label="y=x")
            axes[i].set_xlabel(f"{lbl}{unit}")
            axes[i].set_ylabel(f"{lbr}{unit}")
            axes[i].set_title(col.replace("_", " ").title())
            axes[i].grid(alpha=0.3)
            axes[i].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{PFX}_scatter_cpu_tsa_lut_vs_fpga.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Une seule methode: scatter non applicable.")

In [ ]:
# ==========================================================
# CELLULE 14 — BOXPLOTS Peak Ratio et Winner Margin par SNR
# ==========================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

snrs = sorted(ok["snr_db"].unique())
mths = sorted(ok["method"].unique())
offsets = np.linspace(-0.15, 0.15, len(mths))
colors = ["#4C78A8", "#F58518", "#72B7B2"]

for j, (col, title, ylabel) in enumerate([
    ("peak_ratio", "Peak Ratio par SNR", "Peak Ratio"),
    ("winner_margin_db", "Winner Margin par SNR", "Margin (dB)")
]):
    for k, meth in enumerate(mths):
        lb = ML.get(meth, meth)
        data = [ok[(ok["method"]==meth)&(ok["snr_db"]==s)][col].dropna().values for s in snrs]
        positions = [i + offsets[k] for i in range(len(snrs))]
        bp = axes[j].boxplot(data, positions=positions, widths=0.25,
                             patch_artist=True, showfliers=True)
        for patch in bp["boxes"]:
            patch.set_facecolor(colors[k % len(colors)])
            patch.set_alpha(0.6)
        bp["medians"][0].set_label(lb)

    axes[j].set_xticks(range(len(snrs)))
    axes[j].set_xticklabels([f"{int(s)} dB" for s in snrs])
    axes[j].set_title(title); axes[j].set_ylabel(ylabel)
    axes[j].set_xlabel("SNR")
    axes[j].grid(alpha=.3); axes[j].legend(fontsize=8)
    if col == "winner_margin_db":
        axes[j].axhline(CFG["winner_margin_db_min"], c="red", ls="--", alpha=.7, label="Seuil")

plt.tight_layout()
plt.savefig(OUT_DIR / f"{PFX}_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ==========================================================
# CELLULE 15 — Peak Ratio injecte vs leurres (histogramme)
# ==========================================================
fig, axes = plt.subplots(1, len(sorted(rok["method"].unique())), figsize=(7*len(sorted(rok["method"].unique())), 4.5))
if not hasattr(axes, "__len__"): axes = [axes]

for i, meth in enumerate(sorted(rok["method"].unique())):
    lb = ML.get(meth, meth)
    m_inj = rok[(rok["method"]==meth) & (rok["is_decoy"]==0)]
    m_dec = rok[(rok["method"]==meth) & (rok["is_decoy"]==1)]
    axes[i].hist(m_inj["peak_ratio"].astype(float), bins=30, alpha=0.7, label="PRN injecte", color="#4C78A8")
    axes[i].hist(m_dec["peak_ratio"].astype(float), bins=30, alpha=0.7, label="PRN leurres", color="#F58518")
    axes[i].axvline(CFG["detection_ratio_min"], c="red", ls="--", lw=2, label=f"Seuil={CFG['detection_ratio_min']}")
    axes[i].set_title(f"{lb} — Peak Ratio: Injecte vs Leurres")
    axes[i].set_xlabel("Peak Ratio"); axes[i].set_ylabel("Nombre")
    axes[i].legend(fontsize=8); axes[i].grid(alpha=.3)
    axes[i].set_yscale("log")

plt.tight_layout()
plt.savefig(OUT_DIR / f"{PFX}_peak_ratio_inj_vs_leurres.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ==========================================================
# CELLULE 16 — ANALYSE TEMPS DETAILLEE
# ==========================================================
tcols = ["time_kernel_s", "time_prepare_s", "time_dma_s", "time_ip_wait_s", "time_end_to_end_s"]

# Source prioritaire pour stats globales: winner (ok), fallback: raw_ok
raw_ok = raw[raw["status"] == "ok"].copy() if "status" in raw.columns else raw.copy()

for c in ["time_s", *tcols]:
    if c in ok.columns:
        ok[c] = pd.to_numeric(ok[c], errors="coerce")
    if c in raw_ok.columns:
        raw_ok[c] = pd.to_numeric(raw_ok[c], errors="coerce")

# Pour la decomposition detaillee, on prend ok si disponible, sinon raw_ok
detail_df = ok if any(c in ok.columns and ok[c].notna().any() for c in tcols) else raw_ok
tc = [c for c in tcols if c in detail_df.columns and detail_df[c].notna().any()]

# Tableau detaille
if tc:
    td = detail_df.groupby("method", as_index=False)[tc].agg(["mean", "median", "std", "min", "max"])
    td.columns = ["_".join(c).strip("_") for c in td.columns]
    td.to_csv(OUT_DIR / f"{PFX}_timing_detail.csv", index=False)
    print("=== PROFILING DETAILLE ===")
    display(td.round(6))
else:
    td = pd.DataFrame({
        "message": ["Aucune colonne de timing detaille exploitable dans winner_results.csv ni raw_results.csv"]
    })
    td.to_csv(OUT_DIR / f"{PFX}_timing_detail.csv", index=False)
    print("=== PROFILING DETAILLE ===")
    display(td)

# Tableau CPU vs FPGA
agg_map = {
    "n": ("file", "count"),
}
if "time_s" in ok.columns and ok["time_s"].notna().any():
    agg_map["time_mean_ms"] = ("time_s", lambda s: 1e3 * s.mean())
    agg_map["time_median_ms"] = ("time_s", lambda s: 1e3 * s.median())
    agg_map["time_p95_ms"] = ("time_s", lambda s: 1e3 * np.nanpercentile(s.dropna(), 95) if s.dropna().size else np.nan)
    agg_map["time_p99_ms"] = ("time_s", lambda s: 1e3 * np.nanpercentile(s.dropna(), 99) if s.dropna().size else np.nan)

perf = ok.groupby("method", as_index=False).agg(**agg_map).sort_values("method").reset_index(drop=True)

# e2e_mean_ms: winner si present, sinon fallback raw_ok
if "time_end_to_end_s" in ok.columns and ok["time_end_to_end_s"].notna().any():
    e2e = ok.groupby("method", as_index=False)["time_end_to_end_s"].mean()
elif "time_end_to_end_s" in raw_ok.columns and raw_ok["time_end_to_end_s"].notna().any():
    e2e = raw_ok.groupby("method", as_index=False)["time_end_to_end_s"].mean()
else:
    e2e = pd.DataFrame({"method": perf["method"], "time_end_to_end_s": np.nan})

perf = perf.merge(e2e, on="method", how="left")
perf["e2e_mean_ms"] = 1e3 * perf["time_end_to_end_s"]
perf = perf.drop(columns=["time_end_to_end_s"])

for col in ["time_mean_ms", "time_median_ms", "time_p95_ms", "time_p99_ms", "e2e_mean_ms"]:
    if col not in perf.columns:
        perf[col] = np.nan

ref_label = "Reference CPU"
ref_val = None
for ref_method in ["cpu_tsa_lut", "cpu_lut_angles"]:
    cpu_ref = perf.loc[perf["method"] == ref_method, "time_mean_ms"]
    if not cpu_ref.empty and pd.notna(cpu_ref.iloc[0]):
        ref_val = float(cpu_ref.iloc[0])
        ref_label = ML.get(ref_method, ref_method)
        break
if ref_val is None:
    ref_val = float(perf["time_mean_ms"].dropna().max()) if perf["time_mean_ms"].notna().any() else np.nan

perf["speedup_vs_cpu"] = ref_val / perf["time_mean_ms"]
perf["label"] = perf["method"].map(ML).fillna(perf["method"])
perf.to_csv(OUT_DIR / f"{PFX}_performance.csv", index=False)
print("\n=== PERFORMANCE ===")
print(f"Reference speedup: {ref_label}")
display(perf.round(4))

# Bar chart
colors = ["#4C78A8", "#F58518", "#72B7B2"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

bars = axes[0].bar(perf["label"], perf["time_mean_ms"], color=colors[:len(perf)])
for b, v in zip(bars, perf["time_mean_ms"]):
    if pd.notna(v):
        axes[0].text(b.get_x() + b.get_width() / 2, b.get_height() * 1.02,
                     f"{v:.0f} ms", ha="center", fontsize=10)
axes[0].set_title("Temps moyen (kernel)")
axes[0].set_ylabel("ms")
axes[0].grid(axis="y", alpha=.3)

# Stacked bar: source winner si dispo, sinon raw_ok
stack_source = ok if any(c in ok.columns and ok[c].notna().any() for c in ["time_prepare_s", "time_kernel_s", "time_dma_s"]) else raw_ok
stack_cols = [c for c in ["time_prepare_s", "time_kernel_s", "time_dma_s"] if c in stack_source.columns and stack_source[c].notna().any()]
if stack_cols:
    tm = stack_source.groupby("method", as_index=False)[stack_cols].mean()
else:
    tm = perf[["method"]].copy()
tm["label"] = tm["method"].map(ML).fillna(tm["method"])

smap = {
    "time_prepare_s": ("Preparation", "#4C78A8"),
    "time_kernel_s": ("Kernel", "#F58518"),
    "time_dma_s": ("DMA", "#72B7B2"),
}
x = np.arange(len(tm))
bottoms = np.zeros(len(tm))
for col, (lbl, clr) in smap.items():
    if col not in tm.columns:
        continue
    vals = tm[col].fillna(0).values * 1e3
    axes[1].bar(x, vals, bottom=bottoms, label=lbl, color=clr)
    bottoms += vals
axes[1].set_xticks(x)
axes[1].set_xticklabels(tm["label"])
axes[1].set_title("Decomposition temps moyen")
axes[1].set_ylabel("ms")
if stack_cols:
    axes[1].legend()
axes[1].grid(axis="y", alpha=.3)

plt.tight_layout()
plt.savefig(OUT_DIR / f"{PFX}_timing.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ==========================================================
# CELLULE 17 — GRAPHIQUES PLOTLY INTERACTIFS
# ==========================================================
try:
    import plotly.express as px
    import plotly.io as pio
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    print("plotly non installe. Installe-le avec: pip install plotly")

if HAS_PLOTLY:
    # Renderer robuste pour notebook Jupyter/Cursor
    try:
        pio.renderers.default = "notebook_connected"
    except Exception:
        pass

    # Rendre la cellule autonome si executee seule
    if "OUT_DIR" not in globals():
        OUT_DIR = Path("Resultats validation TSA/analyse_cpu_tsa_lut_vs_fpga_these_vfin")
    if "PFX" not in globals():
        PFX = "cpu_tsa_lut_vs_fpga_these_vfin"
    if "ML" not in globals():
        ML = {
            "cpu_tsa_lut": "CPU TSA LUT",
            "cpu_lut_angles": "CPU TSA LUT",
            "fpga": "FPGA THESE vfin",
        }

    if "ok" not in globals() or ok is None or len(ok) == 0:
        ok = pd.read_csv(OUT_DIR / f"{PFX}_winner_results.csv")
        ok = ok[ok["status"] == "ok"].copy()
    if "rok" not in globals() or rok is None or len(rok) == 0:
        rok = pd.read_csv(OUT_DIR / f"{PFX}_raw_results.csv")
        rok = rok[rok["status"] == "ok"].copy()
        rok["is_decoy"] = (rok["prn_tested"].astype(int) != rok["prn_injected"].astype(int)).astype(int)

    methods = sorted(ok["method"].unique())

    if "ts" not in globals() or ts is None or len(ts) == 0:
        ts = pd.read_csv(OUT_DIR / f"{PFX}_table_by_snr.csv")
    if "pfa_snr" not in globals() or pfa_snr is None or len(pfa_snr) == 0:
        pfa_snr = pd.read_csv(OUT_DIR / f"{PFX}_pfa_by_snr.csv")
    if "perf" not in globals() or perf is None or len(perf) == 0:
        perf = pd.read_csv(OUT_DIR / f"{PFX}_performance.csv")

    ok2 = ok.copy()
    ok2["label"] = ok2["method"].map(ML).fillna(ok2["method"])

    ts2 = ts.copy()
    if "label" not in ts2.columns:
        ts2["label"] = ts2["method"].map(ML).fillna(ts2["method"])

    pfa2 = pfa_snr.copy()
    if "label" not in pfa2.columns:
        pfa2["label"] = pfa2["method"].map(ML).fillna(pfa2["method"])

    perf2 = perf.copy()
    if "label" not in perf2.columns:
        perf2["label"] = perf2["method"].map(ML).fillna(perf2["method"])

    figs = []

    figs.append(px.line(
        ts2, x="snr_db", y="succes_strict_pct", color="label",
        title="Succes strict vs SNR",
        labels={"snr_db": "SNR (dB)", "succes_strict_pct": "Succes (%)"}
    ))

    figs.append(px.line(
        pfa2, x="snr_db", y="Pfa_pct", color="label",
        title="Pfa leurres vs SNR",
        labels={"snr_db": "SNR (dB)", "Pfa_pct": "Pfa (%)"}
    ))

    figs.append(px.histogram(
        ok2[ok2["doppler_err_hz"] <= 2000], x="doppler_err_hz", color="label",
        title="Distribution erreurs Doppler", nbins=40,
        barmode="overlay", opacity=0.7
    ))

    figs.append(px.histogram(
        ok2, x="phase_err_chip", color="label",
        title="Distribution erreurs Phase", nbins=20,
        barmode="overlay", opacity=0.7
    ))

    figs.append(px.bar(
        perf2, x="label", y="time_mean_ms", color="label",
        title="Temps moyen (ms)"
    ))

    if len(methods) >= 2:
        pv_pr = ok2.pivot(index="file", columns="method", values="peak_ratio").dropna().reset_index()
        m0, m1 = methods[0], methods[1]
        if m0 in pv_pr.columns and m1 in pv_pr.columns:
            fig6 = px.scatter(
                pv_pr, x=m0, y=m1,
                title=f"Peak Ratio: {ML.get(m0, m0)} vs {ML.get(m1, m1)}",
                hover_data=["file"]
            )
            mx = max(pv_pr[m0].max(), pv_pr[m1].max())
            fig6.add_shape(type="line", x0=0, y0=0, x1=mx, y1=mx,
                           line=dict(dash="dash", color="red"))
            figs.append(fig6)

    rok2 = rok.copy()
    rok2["type"] = rok2["is_decoy"].map({0: "Injecte", 1: "Leurre"})
    rok2["label"] = rok2["method"].map(ML).fillna(rok2["method"])
    figs.append(px.box(
        rok2, x="label", y="peak_ratio", color="type",
        title="Peak Ratio: Injecte vs Leurres",
        labels={"peak_ratio": "Peak Ratio", "label": "Methode"}
    ))

    for fig in figs:
        fig.show()

In [ ]:
# ==========================================================
# CELLULE 18 — DOMAINE DE VALIDITE
# ==========================================================
domain_rows = []
for meth in sorted(ok["method"].unique()):
    m = ok[ok["method"]==meth]
    ms = m[m["strict_success"]==1]
    if ms.empty:
        domain_rows.append({"method": meth}); continue
    domain_rows.append({
        "method": meth,
        "label": ML.get(meth, meth),
        "snr_min_ok_db": float(ms["snr_db"].min()),
        "snr_max_ok_db": float(ms["snr_db"].max()),
        "doppler_min_ok_hz": int(ms["doppler_true_hz"].min()),
        "doppler_max_ok_hz": int(ms["doppler_true_hz"].max()),
        "n_succes": int(ms.shape[0]),
        "n_total": int(m.shape[0]),
        "taux_succes_pct": round(100.0*ms.shape[0]/m.shape[0], 2),
        "doppler_max_err_hz": int(ms["doppler_err_hz"].max()),
        "phase_max_err_chip": int(ms["phase_err_chip"].max()),
    })
dom = pd.DataFrame(domain_rows)
dom.to_csv(OUT_DIR / f"{PFX}_domaine_validite.csv", index=False)
print("=== DOMAINE DE VALIDITE ===")
display(dom)

In [ ]:
# ==========================================================
# CELLULE 20 — EXPORT WORD (graphiques + tableaux de donnees)
# ==========================================================
try:
    from docx import Document as DocxDocument
    from docx.shared import Inches, Pt, RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.enum.table import WD_TABLE_ALIGNMENT
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-docx"])
    from docx import Document as DocxDocument
    from docx.shared import Inches, Pt, RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.enum.table import WD_TABLE_ALIGNMENT

doc = DocxDocument()
style = doc.styles["Normal"]
style.font.name = "Calibri"
style.font.size = Pt(10)

def _h(text, level=1):
    h = doc.add_heading(text, level=level)
    for r in h.runs:
        r.font.color.rgb = RGBColor(0x1F, 0x3A, 0x5F)

def _tbl(df, max_rows=80, fs=7):
    df = df.head(max_rows).copy()
    for c in df.select_dtypes(include="float").columns:
        df[c] = df[c].apply(lambda v: f"{v:.4f}" if pd.notna(v) else "")
    t = doc.add_table(rows=1+len(df), cols=len(df.columns), style="Light Grid Accent 1")
    t.alignment = WD_TABLE_ALIGNMENT.CENTER
    for j, cn in enumerate(df.columns):
        cell = t.rows[0].cells[j]
        cell.text = str(cn)
        for p in cell.paragraphs:
            p.alignment = WD_ALIGN_PARAGRAPH.CENTER
            for r in p.runs:
                r.bold = True; r.font.size = Pt(fs)
    for i, (_, row) in enumerate(df.iterrows()):
        for j, val in enumerate(row):
            cell = t.rows[i+1].cells[j]
            cell.text = str(val)
            for p in cell.paragraphs:
                for r in p.runs:
                    r.font.size = Pt(fs)

def _img(path, w=6.2):
    if not Path(path).exists():
        return
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p.add_run().add_picture(str(path), width=Inches(w))

# ---------- PAGE TITRE ----------
tp = doc.add_paragraph()
tp.alignment = WD_ALIGN_PARAGRAPH.CENTER
r1 = tp.add_run("Validation de l'IP d'acquisition GNSS\n")
r1.bold = True; r1.font.size = Pt(22); r1.font.color.rgb = RGBColor(0x1F, 0x3A, 0x5F)
r2 = tp.add_run("Analyse comparative CPU TSA LUT vs FPGA THESE vfin\n\n")
r2.font.size = Pt(14); r2.font.color.rgb = RGBColor(0x44, 0x44, 0x44)
r3 = tp.add_run("Rapport genere automatiquement")
r3.font.size = Pt(10); r3.italic = True
doc.add_page_break()

# ---------- 1. PARAMETRES ----------
_h("1. Parametres de campagne")
_tbl(campaign_params, fs=8)
doc.add_paragraph()

# ---------- 2. GLOBAL ----------
_h("2. Resultats globaux par methode")
doc.add_paragraph("Metriques agregees pour chaque methode sur l'ensemble de la campagne.")
_tbl(tg)
doc.add_paragraph()

# ---------- 3. PAR SNR ----------
_h("3. Resultats par SNR")
doc.add_paragraph("Ventilation des performances par niveau de rapport signal-sur-bruit.")
_tbl(ts)
doc.add_paragraph()
_h("3.1. Detection, fausse alarme et miss vs SNR", level=2)
_img(OUT_DIR / f"{PFX}_pd_pfa_pm_vs_snr.png")
doc.add_paragraph()

# ---------- 4. PFA LEURRES ----------
doc.add_page_break()
_h("4. Analyse PFA sur PRN leurres")

_h("4.1. Volume injecte vs leurres", level=2)
_tbl(vol[["method", "type", "n"]], fs=8)
doc.add_paragraph()
_img(OUT_DIR / f"{PFX}_volume_inj_vs_leurres.png")
doc.add_paragraph()

_h("4.2. PFA globale", level=2)
_tbl(pfa_global)
doc.add_paragraph()

_h("4.3. PFA par SNR", level=2)
_tbl(pfa_snr)
doc.add_paragraph()
_img(OUT_DIR / f"{PFX}_pfa_leurres_vs_snr.png")
doc.add_paragraph()

_h("4.4. PFA par PRN leurre", level=2)
_tbl(pfa_prn)
doc.add_paragraph()

_h("4.5. Peak Ratio injecte vs leurres", level=2)
_img(OUT_DIR / f"{PFX}_peak_ratio_inj_vs_leurres.png")
doc.add_paragraph()

# ---------- 5. CONCORDANCE ----------
doc.add_page_break()
_h("5. Concordance CPU TSA LUT vs FPGA THESE vfin")

_h("5.1. Concordance globale", level=2)
_tbl(agr_df)
doc.add_paragraph()

_h("5.2. Concordance par SNR", level=2)
_tbl(agr_snr_df)
doc.add_paragraph()

_h("5.3. Scatter CPU TSA LUT vs FPGA", level=2)
_img(OUT_DIR / f"{PFX}_scatter_cpu_tsa_lut_vs_fpga.png")
doc.add_paragraph()

# ---------- 6. PAR METHODE ----------
doc.add_page_break()
_h("6. Graphiques detailles par methode")
for idx_m, meth in enumerate(sorted(ok["method"].unique())):
    lb = ML.get(meth, meth)
    _h(f"6.{idx_m+1}. {lb}", level=2)
    _img(OUT_DIR / f"{PFX}_method_{meth}.png")
    g = ts[ts["method"] == meth]
    if not g.empty:
        _tbl(g)
    doc.add_paragraph()

# ---------- 7. HEATMAPS ----------
doc.add_page_break()
_h("7. Heatmaps succes strict SNR x Doppler")
for idx_m, meth in enumerate(sorted(ok["method"].unique())):
    lb = ML.get(meth, meth)
    _h(f"7.{idx_m+1}. {lb}", level=2)
    _img(OUT_DIR / f"{PFX}_heatmap_{meth}.png")
    m = ok[ok["method"] == meth]
    pv = m.pivot_table(index="snr_db", columns="doppler_true_hz",
                       values="strict_success", aggfunc="mean") * 100.0
    pve = pv.reset_index().rename(columns={"snr_db": "SNR (dB)"})
    pve.columns = [str(c) for c in pve.columns]
    _tbl(pve.round(1), fs=6)
    doc.add_paragraph()

# ---------- 8. DISTRIBUTIONS ----------
doc.add_page_break()
_h("8. Distributions des erreurs")
_img(OUT_DIR / f"{PFX}_distributions.png")
doc.add_paragraph()

_h("8.1. Statistiques erreurs par methode", level=2)
err_stats = (
    ok.groupby("method", as_index=False)
    .agg(
        doppler_err_mean=("doppler_err_hz", "mean"),
        doppler_err_median=("doppler_err_hz", "median"),
        doppler_err_std=("doppler_err_hz", "std"),
        doppler_err_max=("doppler_err_hz", "max"),
        phase_err_mean=("phase_err_chip", "mean"),
        phase_err_median=("phase_err_chip", "median"),
        phase_err_std=("phase_err_chip", "std"),
        phase_err_max=("phase_err_chip", "max"),
        peak_ratio_mean=("peak_ratio", "mean"),
        peak_ratio_std=("peak_ratio", "std"),
        peak_ratio_min=("peak_ratio", "min"),
        peak_ratio_max=("peak_ratio", "max"),
    )
)
err_stats["label"] = err_stats["method"].map(ML)
_tbl(err_stats)
doc.add_paragraph()

# ---------- 9. BOXPLOTS ----------
_h("9. Boxplots Peak Ratio et Winner Margin par SNR")
_img(OUT_DIR / f"{PFX}_boxplots.png")
box_stats = (
    ok.groupby(["method", "snr_db"], as_index=False)
    .agg(
        peak_ratio_mean=("peak_ratio", "mean"),
        peak_ratio_median=("peak_ratio", "median"),
        peak_ratio_min=("peak_ratio", "min"),
        peak_ratio_max=("peak_ratio", "max"),
        margin_mean_db=("winner_margin_db", "mean"),
        margin_min_db=("winner_margin_db", "min"),
        margin_max_db=("winner_margin_db", "max"),
    ).sort_values(["snr_db", "method"])
)
_tbl(box_stats)
doc.add_paragraph()

# ---------- 10. CAS DIFFICILES ----------
doc.add_page_break()
_h("10. Cas difficiles")
doc.add_paragraph(
    "Signaux tries par echec puis par marge de robustesse croissante. "
    "Ces cas representent les limites du systeme."
)
_tbl(hc, fs=6)
doc.add_paragraph()

# ---------- 11. PERFORMANCE ----------
doc.add_page_break()
_h("11. Analyse de performance")

_h("11.1. Comparaison des temps", level=2)
_tbl(perf)
doc.add_paragraph()
_img(OUT_DIR / f"{PFX}_timing.png")
doc.add_paragraph()

_h("11.2. Profiling detaille", level=2)
_tbl(td.round(6), fs=6)
doc.add_paragraph()

# ---------- 12. DOMAINE VALIDITE ----------
_h("12. Domaine de validite")
doc.add_paragraph(
    "Resume du domaine dans lequel chaque methode produit un resultat "
    "strictement correct (PRN, Doppler et phase dans les tolerances)."
)
_tbl(dom)
doc.add_paragraph()

# ---------- 13. RESUME FINAL ----------
doc.add_page_break()
_h("13. Resume final")

lines = []
lines.append(f"Fichiers signal analyses: {ok['file'].nunique()}")
lines.append(f"SNR testes: {sorted(ok['snr_db'].unique())} dB")
lines.append(f"Plage Doppler: [{int(rok['doppler_true_hz'].min())}, {int(rok['doppler_true_hz'].max())}] Hz")
lines.append(f"PRN injecte: {sorted(rok['prn_injected'].astype(int).unique())}")
lines.append(f"PRN leurres/signal: {rok.groupby(['file','method']).size().iloc[0]-1}")
lines.append("")
for meth in sorted(ok["method"].unique()):
    m = ok[ok["method"] == meth]
    lb = ML.get(meth, meth)
    lines.append(f"--- {lb} ({len(m)} acquisitions) ---")
    lines.append(f"  Succes strict:  {100.0*m['strict_success'].mean():.2f}%")
    lines.append(f"  Pd: {100.0*m['pd_flag'].mean():.2f}%  |  Pfa: {100.0*m['pfa_flag'].mean():.2f}%  |  Pm: {100.0*m['pm_flag'].mean():.2f}%")
    lines.append(f"  Doppler MAE: {np.mean(np.abs(m['doppler_err_hz'])):.2f} Hz")
    lines.append(f"  Phase MAE: {np.mean(np.abs(m['phase_err_chip'])):.2f} chips")
    lines.append(f"  Temps kernel: {1e3*m['time_s'].mean():.1f} ms")
    lines.append("")
for meth in sorted(rok_dec["method"].unique()):
    md = rok_dec[rok_dec["method"] == meth]
    lb = ML.get(meth, meth)
    lines.append(f"--- {lb} PFA leurres ({len(md)} tests) ---")
    lines.append(f"  Fausses detections: {int(md['false_detection'].sum())}")
    lines.append(f"  Pfa: {100.0*md['false_detection'].mean():.2f}%")
    lines.append("")
if not agr_df.empty:
    r = agr_df.iloc[0]
    lines.append(f"--- Concordance {r['gauche']} vs {r['droite']} ---")
    lines.append(f"  Accord decision: {r['accord_decision_pct']:.2f}%")
    lines.append(f"  Accord PRN: {r['accord_prn_pct']:.2f}%")
    lines.append(f"  Accord Doppler exact: {r['accord_doppler_exact_pct']:.2f}%")
    lines.append(f"  Correlation peak ratio: {r['correlation_peak_ratio']:.6f}")
    lines.append("")
if not perf.empty and len(perf) >= 2:
    sp = perf.loc[perf["method"] == "fpga", "speedup_vs_cpu"]
    if not sp.empty:
        lines.append(f"Speedup FPGA vs CPU: {float(sp.iloc[0]):.1f}x")

for line in lines:
    doc.add_paragraph(line)

# ---------- SAVE ----------
docx_path = OUT_DIR / f"{PFX}_rapport_validation.docx"
doc.save(str(docx_path))
print(f"Document Word exporte: {docx_path}")
print(f"Taille: {docx_path.stat().st_size / 1024:.0f} Ko")

In [ ]:
# ==========================================================
# CELLULE 19 — RESUME FINAL
# ==========================================================
errors = W[W["status"] != "ok"]
summary_title = COMPARISON_TITLE if "COMPARISON_TITLE" in globals() else "CPU TSA LUT vs FPGA STSA vfin"

print("=" * 65)
print(f"   RESUME FINAL — VALIDATION {summary_title}")
print("=" * 65)

print(f"\nFichiers signal:        {ok['file'].nunique()}")
print(f"SNR testes:             {sorted(ok['snr_db'].unique())} dB")
print(f"Doppler testes:         [{int(rok['doppler_true_hz'].min())}, {int(rok['doppler_true_hz'].max())}] Hz")
print(f"PRN injecte:            {sorted(rok['prn_injected'].astype(int).unique())}")
print(f"PRN leurres/signal:     {rok.groupby(['file', 'method']).size().iloc[0] - 1}")
print(f"Total lignes raw:       {len(raw)}")
print(f"Total winner:           {len(W)}  (ok: {len(ok)}, erreurs: {len(errors)})")

for meth in sorted(ok["method"].unique()):
    m = ok[ok["method"] == meth]
    lb = ML.get(meth, meth)
    n = len(m)
    ss = 100.0 * m["strict_success"].mean()
    pd_r = 100.0 * m["pd_flag"].mean()
    pfa_r = 100.0 * m["pfa_flag"].mean()
    pm_r = 100.0 * m["pm_flag"].mean()
    d_mae = np.mean(np.abs(m["doppler_err_hz"]))
    p_mae = np.mean(np.abs(m["phase_err_chip"]))
    t_ms = 1e3 * m["time_s"].mean()
    wm = m["winner_margin_db"].mean()

    print(f"\n--- {lb} ({n} acquisitions) ---")
    print(f"  Succes strict:   {ss:.2f}%")
    print(f"  Pd:  {pd_r:.2f}%   |   Pfa: {pfa_r:.2f}%   |   Pm: {pm_r:.2f}%")
    print(f"  Doppler MAE:     {d_mae:.2f} Hz   (max: {int(m['doppler_err_hz'].max())} Hz)")
    print(f"  Phase MAE:       {p_mae:.2f} chips (max: {int(m['phase_err_chip'].max())} chips)")
    print(f"  Peak ratio moy:  {m['peak_ratio'].mean():.2f}  (std: {m['peak_ratio'].std():.2f})")
    print(f"  Winner margin:   {wm:.2f} dB (min: {m['winner_margin_db'].min():.2f} dB)")
    print(f"  Temps kernel:    {t_ms:.1f} ms")

# PFA leurres
for meth in sorted(rok_dec["method"].unique()):
    md = rok_dec[rok_dec["method"] == meth]
    lb = ML.get(meth, meth)
    pfa_l = 100.0 * md["false_detection"].mean()
    print(f"\n--- {lb} — PFA sur {len(md)} tests leurres ---")
    print(f"  Fausses detections:  {int(md['false_detection'].sum())}")
    print(f"  Pfa leurres:         {pfa_l:.2f}%")
    print(f"  Peak ratio max:      {md['peak_ratio'].astype(float).max():.4f}")

# Concordance
if not agr_df.empty:
    r = agr_df.iloc[0]
    print(f"\n--- Concordance {r['gauche']} vs {r['droite']} ---")
    print(f"  Fichiers communs:         {int(r['n_commun'])}")
    print(f"  Accord decision:          {r['accord_decision_pct']:.2f}%")
    print(f"  Accord PRN:               {r['accord_prn_pct']:.2f}%")
    print(f"  Accord Doppler exact:     {r['accord_doppler_exact_pct']:.2f}%")
    print(f"  Accord Phase exact:       {r['accord_phase_exact_pct']:.2f}%")
    print(f"  Delta peak ratio moy:     {r['delta_peak_ratio_moy']:.4f}")
    print(f"  Correlation peak ratio:   {r['correlation_peak_ratio']:.6f}")

# Speedup
if not perf.empty and len(perf) >= 2:
    sp = perf.loc[perf["method"] == "fpga", "speedup_vs_cpu"]
    if not sp.empty:
        print(f"\n--- Performance ---")
        print(f"  Speedup FPGA vs reference CPU:  {float(sp.iloc[0]):.1f}x")

if len(errors) > 0:
    print(f"\n--- Erreurs ({len(errors)}) ---")
    display(errors[["file", "method", "status", "error_message"]].head(10))

print(f"\n{'=' * 65}")
print(f"Tous les exports dans: {OUT_DIR}/")
for f in sorted(OUT_DIR.glob("*.csv")):
    print(f"  {f.name}")
for f in sorted(OUT_DIR.glob("*.png")):
    print(f"  {f.name}")
print(f"{'=' * 65}")
print("FIN DE L'ANALYSE")